### RAG Pipelines - Data Ingestion to vector DB Pipeline

In [37]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [38]:
### reading all the pdf inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF file in a directory"""

    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n processing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_docs.extend(documents)
            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error : {e}")

    print(f"\n Total documents loaded : {len(all_docs)}")
    return all_docs

all_pdf_doc = process_all_pdfs("../data/pdf")

Found 3 PDF files to process

 processing : Devpilot Document.pdf
  Loaded 6 pages

 processing : camerareport.pdf
  Loaded 7 pages

 processing : advancepyrep.pdf
  Loaded 16 pages

 Total documents loaded : 29


In [39]:
all_pdf_doc

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-05-25T06:29:54+00:00', 'moddate': '2026-05-25T06:29:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': '../data/pdf/Devpilot Document.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Devpilot Document.pdf', 'file_type': 'pdf'}, page_content='DevPilot: An Intelligent GitHub-Driven Software\nProject Management and Defect Prediction System\n1st Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n2nd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n3rd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n4th Given Name Surname\ndept. name of organization

In [40]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split document into smaller chunk for bettter RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk : ")
        print(f"Context : {split_docs[0].page_content[:200]} ...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [41]:
chunks = split_documents(all_pdf_doc)
chunks

Split 29 documents into 114 chunks

 Example chunk : 
Context : DevPilot: An Intelligent GitHub-Driven Software
Project Management and Defect Prediction System
1st Given Name Surname
dept. name of organization (of Aff.)
name of organization (of Aff.)
City, Country ...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-05-25T06:29:54+00:00', 'moddate': '2026-05-25T06:29:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': '../data/pdf/Devpilot Document.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Devpilot Document.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-05-25T06:29:54+00:00', 'moddate': '2026-05-25T06:29:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': '../data/pdf/Devpilot Document.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Devpilot Document.pdf', 'file_type': 'pdf'}, page_content='DevPilot: An Intelligent GitHub-Driven Software\nProject Management and Defect Prediction System\n1st Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n2nd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n3rd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n4th Given Name Surname\ndept. name of organization

### Embedding and vectorStoreDB

In [42]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Traformer model"""
        try:
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f" Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embedding with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8362.47it/s]


Model loaded successfully. Embedding dimension : 384


In [47]:
class VectorStore:
    """Manages document embeddings in chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            Collection_name: Name of the ChromaDB collection
            Persist_directory: Directory t persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description" : "PDF deocument embedding for RAG"}
            )
            print(f"Vector store initialized. collection : {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documentss and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding document to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. collection : pdf_documents
Existing document in collection: 0


In [36]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-05-25T06:29:54+00:00', 'moddate': '2026-05-25T06:29:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': '../data/pdf/Devpilot Document.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Devpilot Document.pdf', 'file_type': 'pdf'}, page_content='DevPilot: An Intelligent GitHub-Driven Software\nProject Management and Defect Prediction System\n1st Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n2nd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n3rd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n4th Given Name Surname\ndept. name of organization

In [50]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embedding for 114 texts...


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.35it/s]


Generated embedding with shape: (114, 384)
Adding 114 documents to vector store
Successfully added 114 documents to vector store
Total documents in collection: 114


### Retirever pipeline from vectorStore

In [58]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriver

        Args:
            vector_store: Vector store containing document embedding
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: the search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

            Returns:
                list of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query : '{query}'")
        print(f"Top_k: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank': i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval : {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)


In [59]:
rag_retriever

In [60]:
rag_retriever.retrieve("DevPilot")

Retrieving documents for query : 'DevPilot'
Top_k: 5, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.06it/s]

Generated embedding with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_58946a16_0',
  'content': 'DevPilot: An Intelligent GitHub-Driven Software\nProject Management and Defect Prediction System\n1st Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n2nd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n3rd Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n4th Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n5th Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\n6th Given Name Surname\ndept. name of organization (of Aff.)\nname of organization (of Aff.)\nCity, Country\nemail address or ORCID\nAbstract—Software development teams are majorly depend\nupon col